# Spornado Disease Risk — Exploratory Data Analysis

**Purpose**: Understand the raw Spornado dataset before model training.  
**Audience**: Data scientists and agronomists reviewing data quality and feature distributions.  
**Relation to production code**: This notebook is for *exploration only*.  All feature engineering shown here is formalised in `src/features.py` and consumed by `src/build_crop_models.py` during training.

## Contents
1. Load Data & Overview
2. Feature Engineering
3. Crop Distribution
4. Disease Distribution & Test Results
5. Spore Count Analysis
6. Temporal Patterns
7. Crop–Disease Heatmap
8. Weather Feature Analysis
9. Disease Tolerance Database (Variety Data)
10. EDA Summary

## Disease Triangle Framework
```
        DISEASE RISK
             /\n            /  \n    WEATHER /    \ CROP VARIETY
           /      \n          /________\n       SPORE PRESSURE
```

> **Note on spore count**: Spore count (`spore_count`) is *excluded* from model features.
> The device classifies a trap as positive/negative based on spore detection — using the count
> to predict the label would be target leakage. Models must predict risk from weather signals
> alone so they are useful **before** a trap is read.

In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

DATA_RAW      = '../data/raw/spornado_weather_spore_data.csv'
DIR_PROCESSED = '../data/processed'
DIR_OUTPUTS   = '../outputs'
os.makedirs(DIR_OUTPUTS, exist_ok=True)

print('✓ Setup complete.')

: 

## 1. Load Data & Overview

In [ ]:
df_raw = pd.read_csv(DATA_RAW)
df_raw.columns = df_raw.columns.str.strip()

col_map = {
    'Crop type':'crop_type',      'Result':'result',
    'Result CQ':'result_cq',      'Spore count':'spore_count',
    'Test':'disease_test',        'Start Date':'start_date',
    'End Date':'end_date',        'GPS latitude':'gps_latitude',
    'GPS longitude':'gps_longitude','Customer name':'customer_name',
    'Laboratory name':'lab_name', 'Location Ref':'location_ref',
    'Spornado serial':'spornado_serial','Ref. number':'ref_number',
    'Cassette':'cassette',        'Created by':'created_by',
    'Created At':'created_at',    'Updated By':'updated_by',
    'Updated At':'updated_at',
    '_source':'data_source'
}
df = df_raw.rename(columns={k:v for k,v in col_map.items() if k in df_raw.columns})
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date']   = pd.to_datetime(df['end_date'],   errors='coerce')

# Ensure GPS columns are numeric (strip any residual apostrophe artefacts)
df['gps_latitude']  = pd.to_numeric(
    df['gps_latitude'].astype(str).str.strip().str.lstrip("'"), errors='coerce')
df['gps_longitude'] = pd.to_numeric(
    df['gps_longitude'].astype(str).str.strip().str.lstrip("'"), errors='coerce')

print(f'Shape : {df.shape[0]:,} records  x  {df.shape[1]} columns')
print(f'Dates : {df.start_date.min().date()}  →  {df.start_date.max().date()}')
print()

miss = df.isnull().sum()
miss_df = pd.DataFrame({'Missing': miss, 'Pct%': (miss/len(df)*100).round(1)})
print('Missing values per column:')
print(miss_df[miss_df.Missing > 0].sort_values('Pct%', ascending=False).to_string())

## 2. Feature Engineering

In [ ]:
df['result_cq']       = pd.to_numeric(df['result_cq'], errors='coerce')
df['spore_count']     = pd.to_numeric(df['spore_count'], errors='coerce').clip(lower=0)
df['spore_count_log'] = np.log1p(df['spore_count'])
df['result_clean']    = df['result'].str.lower().str.strip()
df['result_binary']   = (df['result_clean'] == 'positive').astype(int)
df['year']            = df['start_date'].dt.year
df['month']           = df['start_date'].dt.month
df['day_of_year']     = df['start_date'].dt.dayofyear

# Structural missingness flag — NOT a data quality issue.
# All data.xlsx was collected without GPS; weather join requires GPS.
# This flag lets the model learn "no weather" as a feature, not treat it as random noise.
df['weather_available'] = df['temperature_mean_c'].notna().astype(int)

print('✓ Engineered features: spore_count_log | result_binary | year | month | day_of_year | weather_available')
print(f'\nWeather available : {df.weather_available.sum():,} rows  ({df.weather_available.mean()*100:.1f}%)')
print(f'No weather        : {(df.weather_available==0).sum():,} rows  ({(df.weather_available==0).mean()*100:.1f}%)  ← All data.xlsx (no GPS collected)')
print()
preview_cols = ['crop_type','disease_test','result','spore_count','spore_count_log','result_binary','month','weather_available']
print(df[preview_cols].head(6).to_string(index=False))

## 3. Crop Distribution

In [ ]:
crop_counts = df['crop_type'].value_counts()
print(f'Unique crops: {len(crop_counts)}\n')
print(crop_counts.to_string())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Crop Type Distribution', fontsize=14, fontweight='bold')

# Bar chart
crop_counts.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='black', alpha=0.8)
axes[0].set_xlabel('Number of Tests')
axes[0].set_title('All Crops — Test Count')
axes[0].grid(axis='x', alpha=0.3)
for i, v in enumerate(crop_counts.values):
    axes[0].text(v + 20, i, f'{v:,}', va='center', fontsize=9)

# Pie chart
top6 = crop_counts.head(6).copy()
top6['Other'] = crop_counts[6:].sum()
axes[1].pie(top6, labels=top6.index, autopct='%1.1f%%',
            colors=sns.color_palette('husl', len(top6)), startangle=140)
axes[1].set_title('Share of Tests (Top 6 + Other)')

plt.tight_layout()
plt.savefig(f'{DIR_OUTPUTS}/01_crop_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: 01_crop_distribution.png')

## 4. Disease Distribution & Test Results

In [ ]:
disease_counts = df['disease_test'].value_counts()
result_counts  = df['result_clean'].value_counts()
print(f'Unique diseases : {len(disease_counts)}')
print(f'Positive rate   : {df.result_binary.mean()*100:.1f}%\n')
print(disease_counts.head(15).to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Disease Distribution & Test Results', fontsize=14, fontweight='bold')

# Top 15 diseases
top15 = disease_counts.head(15)
sns.barplot(x=top15.values, y=top15.index, palette='viridis', ax=axes[0])
axes[0].set_title('Top 15 Diseases Tested')
axes[0].set_xlabel('Test Count')
axes[0].grid(axis='x', alpha=0.3)
for i, v in enumerate(top15.values):
    axes[0].text(v + 5, i, str(v), va='center', fontsize=8)

# Result distribution
colors_r = ['#2ecc71', '#e74c3c', '#95a5a6']
bars = axes[1].bar(result_counts.index, result_counts.values,
                   color=colors_r[:len(result_counts)], edgecolor='black', alpha=0.85)
axes[1].set_title('Positive vs Negative Results')
axes[1].set_ylabel('Count')
axes[1].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, result_counts.values):
    pct = val / result_counts.sum() * 100
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 50,
                 f'{pct:.1f}%\n({val:,})', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{DIR_OUTPUTS}/02_disease_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: 02_disease_distribution.png')

## 5. Spore Count Analysis

In [ ]:
print('Spore Count Statistics:')
print(df['spore_count'].describe().round(1).to_string())
print(f'\nZero spore count : {(df.spore_count==0).sum():,}')
print(f'Missing          : {df.spore_count.isna().sum():,}')
print(f'Very high (>10K) : {(df.spore_count>10000).sum():,}')

def spore_cat(v):
    if pd.isna(v):  return 'Missing'
    if v == 0:      return 'Zero'
    if v < 100:     return 'Low (1-99)'
    if v < 1000:    return 'Medium (100-999)'
    if v < 10000:   return 'High (1K-9.9K)'
    return 'Very High (10K+)'

df['spore_cat'] = df['spore_count'].apply(spore_cat)
order = ['Zero','Low (1-99)','Medium (100-999)','High (1K-9.9K)','Very High (10K+)','Missing']
cats  = df['spore_cat'].value_counts().reindex([c for c in order if c in df['spore_cat'].unique()])
spore_pos = df[df['spore_count'] > 0].copy()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Spore Count Analysis', fontsize=14, fontweight='bold')

# Log distribution
axes[0,0].hist(np.log1p(spore_pos['spore_count']), bins=40,
               color='steelblue', edgecolor='black', alpha=0.75)
axes[0,0].set_title('Log(Spore Count) Distribution')
axes[0,0].set_xlabel('log1p(Spore Count)')
axes[0,0].set_ylabel('Frequency')
axes[0,0].grid(alpha=0.3)

# Boxplot by result
sns.boxplot(data=spore_pos, x='result_clean', y='spore_count', ax=axes[0,1],
            palette={'positive':'#e74c3c', 'negative':'#2ecc71'})
axes[0,1].set_yscale('log')
axes[0,1].set_title('Spore Count by Test Result (log scale)')
axes[0,1].set_ylabel('Spore Count')
axes[0,1].grid(axis='y', alpha=0.3)

# Category bar
clrs = ['#95a5a6','#f39c12','#e67e22','#e74c3c','#c0392b','#34495e']
axes[1,0].bar(range(len(cats)), cats.values, color=clrs[:len(cats)], edgecolor='black', alpha=0.85)
axes[1,0].set_xticks(range(len(cats)))
axes[1,0].set_xticklabels(cats.index, rotation=30, ha='right')
axes[1,0].set_title('Spore Count Categories')
axes[1,0].set_ylabel('Records')
axes[1,0].grid(axis='y', alpha=0.3)
for i, v in enumerate(cats.values):
    axes[1,0].text(i, v+30, f'{v:,}', ha='center', fontsize=8, fontweight='bold')

# Top 10 highest spore counts by crop
top10 = df.nlargest(10, 'spore_count')[['crop_type','spore_count']]
axes[1,1].bar(range(10), top10['spore_count'].values, color='darkred', edgecolor='black', alpha=0.8)
axes[1,1].set_xticks(range(10))
axes[1,1].set_xticklabels(top10['crop_type'].values, rotation=40, ha='right')
axes[1,1].set_title('Top 10 Highest Spore Counts by Crop')
axes[1,1].set_ylabel('Spore Count')
axes[1,1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DIR_OUTPUTS}/03_spore_count_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: 03_spore_count_analysis.png')

## 6. Temporal Patterns

In [ ]:
df_t = df.dropna(subset=['start_date'])
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

daily       = df_t.groupby(df_t['start_date'].dt.date).size()
monthly_cnt = df_t['month'].value_counts().sort_index()
yearly_cnt  = df_t['year'].value_counts().sort_index()
monthly_pos = df_t.groupby('month')['result_binary'].mean() * 100

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Temporal Patterns', fontsize=14, fontweight='bold')

# Daily volume
axes[0,0].plot(daily.index, daily.values, color='steelblue', linewidth=1.2, alpha=0.8)
axes[0,0].fill_between(daily.index, daily.values, alpha=0.2, color='steelblue')
axes[0,0].set_title('Daily Test Volume Over Time')
axes[0,0].set_ylabel('Tests per Day')
axes[0,0].grid(alpha=0.3)
axes[0,0].tick_params(axis='x', rotation=30)

# By month
axes[0,1].bar(monthly_cnt.index, monthly_cnt.values, color='teal', edgecolor='black', alpha=0.8)
axes[0,1].set_xticks(range(1,13))
axes[0,1].set_xticklabels(month_labels, rotation=40, ha='right')
axes[0,1].set_title('Tests by Month')
axes[0,1].set_ylabel('Count')
axes[0,1].grid(axis='y', alpha=0.3)

# By year
axes[1,0].bar(yearly_cnt.index.astype(str), yearly_cnt.values,
              color='coral', edgecolor='black', alpha=0.8)
axes[1,0].set_title('Tests by Year')
axes[1,0].set_ylabel('Count')
axes[1,0].grid(axis='y', alpha=0.3)
for i, v in enumerate(yearly_cnt.values):
    axes[1,0].text(i, v+30, f'{v:,}', ha='center', fontsize=9)

# Positive rate by month
axes[1,1].plot(monthly_pos.index, monthly_pos.values,
               marker='o', color='darkred', linewidth=2, markersize=7)
axes[1,1].fill_between(monthly_pos.index, monthly_pos.values, alpha=0.2, color='darkred')
axes[1,1].set_xticks(range(1,13))
axes[1,1].set_xticklabels(month_labels, rotation=40, ha='right')
axes[1,1].set_title('Disease Positive Rate by Month')
axes[1,1].set_ylabel('Positive Rate (%)')
axes[1,1].set_ylim(0, 100)
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DIR_OUTPUTS}/04_temporal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: 04_temporal_patterns.png')

## 7. Crop–Disease Heatmap

In [ ]:
top_crops    = df['crop_type'].value_counts().head(6).index
top_diseases = df['disease_test'].value_counts().head(8).index
ct = pd.crosstab(
    df[df['crop_type'].isin(top_crops)]['crop_type'],
    df[df['disease_test'].isin(top_diseases)]['disease_test']
)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Crop–Disease Relationships', fontsize=14, fontweight='bold')

# Heatmap
sns.heatmap(ct, annot=True, fmt='d', cmap='YlOrRd', ax=axes[0],
            cbar_kws={'label': 'Test Count'})
axes[0].set_title('Top 6 Crops vs Top 8 Diseases')
plt.setp(axes[0].get_xticklabels(), rotation=40, ha='right', fontsize=8)

# Top 10 combinations
ct_flat = ct.stack().sort_values(ascending=False).head(10).reset_index()
ct_flat.columns = ['Crop', 'Disease', 'Count']
ct_flat['Label'] = ct_flat['Crop'] + ' — ' + ct_flat['Disease']
sns.barplot(data=ct_flat, x='Count', y='Label', palette='muted', ax=axes[1])
axes[1].set_title('Top 10 Crop–Disease Combinations')
axes[1].set_xlabel('Test Count')
axes[1].grid(axis='x', alpha=0.3)
for i, v in enumerate(ct_flat['Count']):
    axes[1].text(v + 5, i, str(v), va='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{DIR_OUTPUTS}/05_crop_disease_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: 05_crop_disease_heatmap.png')

### 7.1 Positive Sample Sufficiency
Which crop–disease combinations have enough positive samples to train a reliable model?  
Combinations with fewer than 10 positives are not viable targets — flag or drop them before committing to an architecture.

In [ ]:
summary = df.groupby(['crop_type', 'disease_test']).agg(
    total=('result_binary', 'count'),
    positives=('result_binary', 'sum'),
    positive_rate=('result_binary', 'mean')
).sort_values('positives', ascending=False)

viable   = summary[summary['positives'] >= 10]
marginal = summary[(summary['positives'] > 0) & (summary['positives'] < 10)]
zero_pos = summary[summary['positives'] == 0]

print(f'Viable combinations (≥10 positives) : {len(viable)}')
print(f'Marginal combinations (1–9 positives): {len(marginal)}')
print(f'No positives at all                  : {len(zero_pos)}\n')
print('--- Viable targets ---')
print(viable.to_string())

## 8. Weather Feature Analysis

**Why weather is partially missing — and why that's OK.**

`All data.xlsx` was provided by the company without GPS coordinates. Since the weather join uses GPS to match each sample to the nearest weather station, all xlsx rows will have empty weather columns. This is **structural missingness by source**, not a data quality problem.

| Source | GPS | Weather joinable |
|---|---|---|
| `All data.xlsx` | No GPS collected | No |
| `More Data.csv` | GPS included | Yes (within 15 km of a station) |

**Strategy for modelling:**
1. Keep all rows — dropping xlsx rows loses ~57% of the training data.
2. Use `weather_available` (engineered in Section 2) as an explicit feature so the model knows when weather context is absent.
3. Use median imputation for weather columns, or use XGBoost / LightGBM which handle `NaN` natively without imputation.
4. Do **not** impute weather values for xlsx rows — there is no spatial basis for those values.

In [ ]:
WEATHER_COLS = ['temperature_min_c','temperature_max_c','temperature_mean_c',
 'humidity_max_percent','precipitation_mm','wind_speed_max_kmh','dew_point_min_c']

df_wx = df.dropna(subset=['temperature_mean_c']).copy()
print(f'Records with weather : {len(df_wx):,} / {len(df):,} ({len(df_wx)/len(df)*100:.1f}%)')
print(f'Unique GPS locations : {df_wx[["gps_latitude","gps_longitude"]].drop_duplicates().shape[0]}')
print()
print('Weather feature summary:')
print(df_wx[WEATHER_COLS].describe().round(2).to_string())

labels = ['Temp Min (C)','Temp Max (C)','Temp Mean (C)',
          'Humidity Max (%)','Precipitation (mm)','Wind Max (km/h)','Dew Point Min (C)']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('Weather Feature Distributions', fontsize=14, fontweight='bold')
for ax, col, lbl in zip(axes.flat, WEATHER_COLS, labels):
    ax.hist(df_wx[col].dropna(), bins=40, color='steelblue', edgecolor='black', alpha=0.7)
    ax.set_title(lbl, fontsize=9)
    ax.set_ylabel('Count')
    ax.grid(alpha=0.3)
axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.savefig(f'{DIR_OUTPUTS}/06_weather_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 06_weather_distributions.png')


In [ ]:
# Weather conditions: positive vs negative tests
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Weather at Test Date: Positive vs Negative', fontsize=13, fontweight='bold')

for ax, col, lbl in zip(
    axes,
    ['temperature_mean_c', 'humidity_max_percent', 'precipitation_mm'],
    ['Mean Temp (C)', 'Max Humidity (%)', 'Precipitation (mm)']
):
    for res, color in [('positive','#e74c3c'), ('negative','#2ecc71')]:
        vals = df_wx[df_wx['result_clean'] == res][col].dropna()
        ax.hist(vals, bins=35, alpha=0.55, color=color, edgecolor='black', label=res)
    ax.set_title(lbl)
    ax.set_ylabel('Count')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DIR_OUTPUTS}/07_weather_vs_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 07_weather_vs_result.png')


## 9. Disease Tolerance Database (Variety Data)

In [ ]:
tol_files = {
    'Corn':    f'{DIR_PROCESSED}/corn_disease_tolerance.csv',
    'Soybean': f'{DIR_PROCESSED}/soybean_disease_tolerance.csv',
    'Potato':  f'{DIR_PROCESSED}/potato_disease_tolerance.csv',
}
tol_dbs = {}
for crop, path in tol_files.items():
    try:
        t = pd.read_csv(path)
        tol_dbs[crop] = t
        diseases = [c for c in t.columns if c != 'Variety']
        print(f'{crop:8s}: {len(t):2d} varieties | Diseases tracked: {diseases}')
    except FileNotFoundError:
        print(f'{crop}: file not found at {path}')

if tol_dbs:
    first = list(tol_dbs.keys())[0]
    print(f'\nSample — {first}:')
    print(tol_dbs[first].head().to_string(index=False))

## 10. EDA Summary

In [ ]:
gps_cov      = df[['gps_latitude','gps_longitude']].notna().all(axis=1).mean() * 100
completeness = (1 - df.isnull().sum().sum() / (len(df) * df.shape[1])) * 100

print('=' * 60)
print('EDA SUMMARY')
print('=' * 60)
print(f'  Records           : {len(df):,}')
print(f'  Date range        : {df.start_date.min().date()} — {df.start_date.max().date()}')
print(f'  Unique crops      : {df.crop_type.nunique()}')
print(f'  Unique diseases   : {df.disease_test.nunique()}')
print(f'  Positive rate     : {df.result_binary.mean()*100:.1f}%')
print(f'  GPS coverage      : {gps_cov:.1f}%  (More Data.csv only — xlsx had no GPS)')
wx_rows = df['weather_available'].sum()
wx_cov  = df['weather_available'].mean() * 100
print(f'  Weather coverage  : {wx_cov:.1f}%  ({wx_rows:,} rows) — structural: xlsx source has no GPS')
print(f'  Data completeness : {completeness:.1f}%')
print()
print('KEY INSIGHTS:')
print(f'  Top crop     : {crop_counts.index[0]}  ({crop_counts.iloc[0]:,} tests)')
print(f'  Top disease  : {disease_counts.index[0]}  ({disease_counts.iloc[0]:,} tests)')
print(f'  High spore (>10K) : {(df.spore_count > 10000).sum():,} records')
print(f'  Zero spore        : {(df.spore_count == 0).sum():,} records (background noise)')
print()
print('WEATHER MISSINGNESS — BY DESIGN:')
src_wx = df.groupby('data_source')['weather_available'].agg(['sum','count'])
src_wx['pct'] = (src_wx['sum'] / src_wx['count'] * 100).round(1)
print(src_wx.rename(columns={'sum':'with_weather','count':'total'}).to_string())
print()
print('  Model strategy: keep all rows | weather_available flag | NaN-native model or median imputation')
print()
print('DISEASE TRIANGLE STATUS:')
print('  [DONE] Spore Pressure  — available in dataset')
print('  [DONE] Temporal/Season — engineered from dates')
print('  [PART] Weather         — available for More_Data_csv rows (~43%); xlsx rows have no GPS by design')
print('  [PEND] Crop Variety    — tolerance DB ready; awaiting farmer records')
print()
print('OUTPUTS SAVED:')
for i, name in enumerate(['01_crop_distribution','02_disease_distribution',
                           '03_spore_count_analysis','04_temporal_patterns',
                           '05_crop_disease_heatmap',
                           '06_weather_distributions','07_weather_vs_result'], 1):
    print(f'  {i}. {DIR_OUTPUTS}/{name}.png')
print()
print('NEXT STEP → Open 02_disease_prediction_model.ipynb')